# Eredivisie 2025–2026 metrics by category

This notebook combines the existing Eredivisie event and model aggregates, derives player positions, prepares category tables, and creates a publication-ready Excel workbook in the Waltzing Analytics Meridian house style.

Position derivation uses dominant lineup position code weighted by stint minutes, formation slot, median on-ball location, goalkeeper evidence, and a touch-sample confidence adjustment. The output includes broad position, detailed position, secondary position, confidence, source, and versatility.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('/Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026')
SCRIPTS = ROOT / 'Scripts'
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

from eredivisie_metrics_by_category_pipeline import (
    build_workbook,
    OUTPUT_XLSX,
    AGGREGATED,
)

OUTPUT_XLSX

PosixPath('/Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026/Aggregated/eredivisie_2025_2026_metrics_by_category.xlsx')

## Build the category workbook and position table

The Python pipeline prepares auditable CSV tables. Workbook authoring is delegated to `Scripts/build_eredivisie_metrics_workbook.mjs`, which uses the bundled `@oai/artifact-tool` runtime. The generated workbook opens with a professional information page covering scope, usage, navigation, methodology, reading conventions, quality notes, and provenance.

In [2]:
result = build_workbook()
result

(node:68830) Warning: The 'NO_COLOR' env is ignored due to the 'FORCE_COLOR' env being set.
(Use `node --trace-warnings ...` to show where the warning was created)


Inspect result written to file: /Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026/Aggregated/eredivisie_2025_2026_metrics_by_category.xlsx.inspect.ndjson
{"output":"/Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026/Aggregated/eredivisie_2025_2026_metrics_by_category.xlsx","sheets":17}


{'manifest': PosixPath('/Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026/Aggregated/NotebookBuild/workbook_manifest.json'),
 'workbook': PosixPath('/Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026/Aggregated/eredivisie_2025_2026_metrics_by_category.xlsx'),
 'positions': PosixPath('/Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026/Aggregated/eredivisie_2025_2026_derived_positions.csv'),
 'player_rows': 554,
 'team_rows': 18,
 'sheets': 17}

## Inspect derived positions

In [3]:
positions = pd.read_csv(result['positions'])
position_summary = (
    positions.groupby(['position_group', 'position_detail'], dropna=False)
    .agg(players=('player_id', 'nunique'), minutes=('minutes', 'sum'), mean_confidence=('position_confidence', 'mean'))
    .reset_index()
    .sort_values(['position_group', 'minutes'], ascending=[True, False])
)
position_summary

,position_group,position_detail,players,minutes,mean_confidence
1,Defender,Left Back,75,126986.0,0.879556
2,Defender,Right Back,66,125620.0,0.946435
0,Defender,Centre Back,19,36975.0,0.901520
3,Forward,Centre Forward,66,108386.0,0.886475
4,Forward,Left Winger,17,21413.0,0.786384
5,Forward,Right Winger,10,16286.0,0.855422
6,Goalkeeper,Goalkeeper,47,77697.0,0.991409
8,Midfielder,Central Midfielder,76,149177.0,0.927737
7,Midfielder,Attacking Midfielder,56,94347.0,0.808757
10,Midfielder,Left Midfielder,43,71544.0,0.842539


In [4]:
low_confidence = positions.loc[
    positions.position_confidence < 0.60,
    ['team', 'player', 'minutes', 'position_group', 'position_detail', 'position_confidence', 'position_source']
].sort_values(['minutes', 'position_confidence'], ascending=[False, True])
low_confidence.head(30)

,team,player,minutes,position_group,position_detail,position_confidence,position_source
224,NAC Breda,D. Versluis,589.0,Midfielder,Left Midfielder,0.524897,lineup position code + touch geography
110,Fortuna Sittard,A. Halilović,477.0,Midfielder,Left Midfielder,0.547568,lineup position code + touch geography
414,FC Groningen,N. Emeran,388.0,Midfielder,Attacking Midfielder,0.576073,lineup position code + touch geography
408,FC Groningen,N. Eggens,292.0,Midfielder,Central Midfielder,0.583104,lineup position code + touch geography
315,Nijmegen Eendracht Combinatie,I. Hansen-Aarøen,289.0,Forward,Centre Forward,0.562388,lineup position code + touch geography
225,NAC Breda,C. Staring,195.0,Forward,Centre Forward,0.513355,lineup position code + touch geography
191,SC Heerenveen,I. Ahmed,194.0,Forward,Left Winger,0.517413,lineup position code + touch geography
331,Nijmegen Eendracht Combinatie,Y. Sbai,192.0,Defender,Left Back,0.560299,lineup position code + touch geography
478,Heracles Almelo,T. Hamamioglu,189.0,Forward,Left Winger,0.587723,lineup position code + touch geography
198,SC Heerenveen,C. Bonevacia,96.0,Defender,Left Back,0.003125,touch geography fallback


## Workbook QA

Confirm that the expected 19-sheet topology was created, including the professional information page, role-specific radar definitions, player-level radar data, and core player position columns.

In [5]:
expected_sheets = [
    'Info', 'Player Standard', 'Player Shooting', 'Player Passing', 'Player Possession',
    'Player Progression', 'Player Defending', 'Player Goalkeeping', 'Player GDA',
    'Player xT Danger', 'Player Impact', 'Profile Definitions', 'Profile Radar Data',
    'Team Standard', 'Team Attack',
    'Team Passing', 'Team Possession', 'Team Defending', 'Metric Dictionary',
]
with pd.ExcelFile(OUTPUT_XLSX) as book:
    actual_sheets = book.sheet_names
assert actual_sheets == expected_sheets, (actual_sheets, expected_sheets)
info_preview = pd.read_excel(OUTPUT_XLSX, sheet_name='Info', header=None)
assert info_preview.astype(str).apply(lambda column: column.str.contains('DATASET SNAPSHOT', na=False)).any().any()

standard = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Standard')
shooting = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Shooting')
passing = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Passing')
possession = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Possession')
progression = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Progression')
defending = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Defending')
goalkeeping = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Goalkeeping')
gda = pd.read_excel(OUTPUT_XLSX, sheet_name='Player GDA')
xt_danger = pd.read_excel(OUTPUT_XLSX, sheet_name='Player xT Danger')
impact = pd.read_excel(OUTPUT_XLSX, sheet_name='Player Impact')
dictionary = pd.read_excel(OUTPUT_XLSX, sheet_name='Metric Dictionary')
profile_definitions = pd.read_excel(OUTPUT_XLSX, sheet_name='Profile Definitions')
profile_radar_data = pd.read_excel(OUTPUT_XLSX, sheet_name='Profile Radar Data')
required_position_columns = {
    'position_group', 'position_detail', 'secondary_position',
    'position_confidence', 'position_source', 'position_versatility',
}
assert required_position_columns.issubset(standard.columns)
assert standard.position_group.notna().mean() > 0.99
required_shooting_columns = {
    'goals', 'goals_per90', 'non_penalty_goals', 'non_penalty_goals_per90',
    'shots', 'shots_per90', 'shots_on_target', 'shots_on_target_per90',
    'danger_xg', 'danger_xg_per90', 'non_penalty_xg', 'non_penalty_xg_per90',
    'box_shot_share', 'big_chance_shot_share', 'goals_per_shot_on_target',
}
assert required_shooting_columns.issubset(shooting.columns)
required_passing_columns = {
    'passes_attempted', 'passes_attempted_per90',
    'passes_completed', 'passes_completed_per90',
    'forward_passes', 'forward_passes_per90', 'forward_pass_completion_pct',
    'long_distance_passes', 'long_distance_passes_per90', 'long_pass_completion_pct',
    'progressive_passes', 'progressive_passes_per90',
    'pressured_pass_completion_pct', 'expected_assists', 'expected_assists_per90',
    'second_assists', 'second_assists_per90', 'third_assists', 'third_assists_per90',
}
assert required_passing_columns.issubset(passing.columns)
required_possession_columns = {
    'events', 'events_per90', 'non_turnover_actions', 'non_turnover_actions_per90',
    'take_ons', 'take_ons_per90', 'take_ons_unsuccessful',
    'turnovers_per_100_actions', 'dangerous_turnover_share',
    'xt_carries', 'xt_carries_per90', 'carries', 'carries_per90',
    'progressive_carries', 'carries_into_final_third', 'carries_into_box',
}
assert required_possession_columns.issubset(possession.columns)
required_progression_columns = {
    'progressive_passes', 'progressive_passes_per90',
    'progressive_pass_distance', 'progressive_pass_distance_per90',
    'passes_into_final_third', 'passes_into_final_third_per90',
    'passes_into_box', 'passes_into_box_per90',
    'xt_added', 'xt_added_per90', 'xt_passes', 'xt_passes_per90',
    'xt_from_all_passes', 'xt_from_progressive_passes_est',
    'xt_per_pass', 'xt_per_progressive_pass_est',
    'second_assists', 'third_assists',
}
assert required_progression_columns.issubset(progression.columns)
assert {'high_defensive_action_share', 'errors_per_100_def_actions'}.issubset(defending.columns)
assert {'goals_prevented_xgot_per_shot', 'long_pass_completion_pct'}.issubset(goalkeeping.columns)
assert {'gda_reliability_adjusted_per90', 'raw_plus_minus_per90'}.issubset(gda.columns)
assert {'xt_passes', 'xt_carries', 'xt_take_ons'}.issubset(xt_danger.columns)
assert {'balanced_impact_score', 'impact_sample_reliability'}.issubset(impact.columns)
assert dictionary.description.notna().all() and dictionary.family.notna().all()
assert profile_definitions.groupby('profile').size().eq(8).all()
assert profile_definitions.reference_players.gt(0).all()
assert profile_radar_data.groupby(['team', 'player_id', 'profile']).size().eq(8).all()
assert profile_radar_data.percentile.dropna().between(0, 100).all()

{
    'workbook': str(OUTPUT_XLSX),
    'sheets': len(actual_sheets),
    'player_rows': len(standard),
    'position_coverage': standard.position_group.notna().mean(),
    'mean_position_confidence': standard.position_confidence.mean(),
}

{'workbook': '/Users/marclamberts/Event data/data/competitions/Eredivisie 2025-2026/Aggregated/eredivisie_2025_2026_metrics_by_category.xlsx',
 'sheets': 17,
 'player_rows': 554,
 'position_coverage': np.float64(1.0),
 'mean_position_confidence': np.float64(0.8921215427352444)}